In [11]:
import os
import time
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    roc_auc_score
)

import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [12]:
import io

with open("original_RT_IoT2022.csv", "r", encoding="utf-8") as f:
    text = f.read()

# Remove quotation marks
text = text.replace('"', '')

# Fix leading comma in header if present
lines = text.splitlines()

if lines[0].startswith(","):
    lines[0] = lines[0][1:]

text = "\n".join(lines)

# Load dataset
df = pd.read_csv(io.StringIO(text))

print("Original dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Original dataset shape: (123117, 84)

Columns:
['id.orig_p', 'id.resp_p', 'proto', 'service', 'flow_duration', 'fwd_pkts_tot', 'bwd_pkts_tot', 'fwd_data_pkts_tot', 'bwd_data_pkts_tot', 'fwd_pkts_per_sec', 'bwd_pkts_per_sec', 'flow_pkts_per_sec', 'down_up_ratio', 'fwd_header_size_tot', 'fwd_header_size_min', 'fwd_header_size_max', 'bwd_header_size_tot', 'bwd_header_size_min', 'bwd_header_size_max', 'flow_FIN_flag_count', 'flow_SYN_flag_count', 'flow_RST_flag_count', 'fwd_PSH_flag_count', 'bwd_PSH_flag_count', 'flow_ACK_flag_count', 'fwd_URG_flag_count', 'bwd_URG_flag_count', 'flow_CWR_flag_count', 'flow_ECE_flag_count', 'fwd_pkts_payload.min', 'fwd_pkts_payload.max', 'fwd_pkts_payload.tot', 'fwd_pkts_payload.avg', 'fwd_pkts_payload.std', 'bwd_pkts_payload.min', 'bwd_pkts_payload.max', 'bwd_pkts_payload.tot', 'bwd_pkts_payload.avg', 'bwd_pkts_payload.std', 'flow_pkts_payload.min', 'flow_pkts_payload.max', 'flow_pkts_payload.tot', 'flow_pkts_payload.avg', 'flow_pkts_payload.std', 'fwd_iat

In [13]:
cols_to_drop = [
    "Unnamed: 0",
    "Flow_ID",
    "Source_IP",
    "Destination_IP",
    "Timestamp",
    "id.orig_p",
    "id.resp_p"
]

df = df.drop(
    columns=[c for c in cols_to_drop if c in df.columns],
    errors="ignore"
)

print("Shape after removing unnecessary columns:", df.shape)

Shape after removing unnecessary columns: (123117, 82)


In [14]:
feat_cols_for_dedup = [
    c for c in df.columns
    if c != "Attack_type"
]

before = len(df)

df = (
    df.drop_duplicates(subset=feat_cols_for_dedup)
      .reset_index(drop=True)
)

after = len(df)

print("=" * 60)
print("DATASET DEDUPLICATION")
print("=" * 60)
print("Rows before deduplication :", before)
print("Rows after deduplication  :", after)
print("Duplicate rows removed    :", before - after)

DATASET DEDUPLICATION
Rows before deduplication : 123117
Rows after deduplication  : 18272
Duplicate rows removed    : 104845


In [15]:
normal_traffic = [
    "Thing_Speak",
    "Wipro_bulb",
    "MQTT_Publish"
]

if df["Attack_type"].dtype == object:

    df["Attack_type"] = df["Attack_type"].apply(
        lambda x: 0 if x in normal_traffic else 1
    )

    print("Attack_type converted to binary.")

else:
    print("Attack_type is already numeric.")

Attack_type converted to binary.


In [16]:
label_encoders = {}

for col in df.select_dtypes(include=["object"]).columns:

    if col == "Attack_type":
        continue

    le = LabelEncoder()

    df[col] = le.fit_transform(
        df[col].astype(str)
    )

    label_encoders[col] = le

X = df.drop(columns=["Attack_type"])
y = df["Attack_type"]

print("=" * 60)
print("FEATURE / TARGET INFORMATION")
print("=" * 60)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nNumber of classes:", y.nunique())

FEATURE / TARGET INFORMATION
X shape: (18272, 81)
y shape: (18272,)

Target distribution:
Attack_type
0    11959
1     6313
Name: count, dtype: int64

Number of classes: 2


In [18]:
# ---------------------------------------------------------
# STEP 1: 80% temporary training pool + 20% final test
# ---------------------------------------------------------

X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

# ---------------------------------------------------------
# STEP 2: Split remaining 80% into:
# 75% of 80% = 60% total training
# 25% of 80% = 20% total validation
# ---------------------------------------------------------

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("=" * 60)
print("DATASET SPLIT")
print("=" * 60)

print(f"Total samples      : {len(X)}")
print(f"Training samples   : {len(X_train)}")
print(f"Validation samples : {len(X_val)}")
print(f"Test samples       : {len(X_test)}")

print("\nPercentages:")
print(f"Training   : {len(X_train)/len(X)*100:.2f}%")
print(f"Validation : {len(X_val)/len(X)*100:.2f}%")
print(f"Test       : {len(X_test)/len(X)*100:.2f}%")

DATASET SPLIT
Total samples      : 18272
Training samples   : 10962
Validation samples : 3655
Test samples       : 3655

Percentages:
Training   : 59.99%
Validation : 20.00%
Test       : 20.00%


In [19]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

X_test_scaled = scaler.transform(X_test)

print("Scaling completed.")
print("Training scaled shape  :", X_train_scaled.shape)
print("Validation scaled shape:", X_val_scaled.shape)
print("Test scaled shape      :", X_test_scaled.shape)

Scaling completed.
Training scaled shape  : (10962, 81)
Validation scaled shape: (3655, 81)
Test scaled shape      : (3655, 81)


In [20]:
rf_param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [5, 10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced"
    ),
    param_distributions=rf_param_dist,
    n_iter=25,
    cv=3,
    scoring="f1_macro",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("Training Random Forest...")
rf_search.fit(X_train, y_train)

best_rf = rf_search.best_estimator_

print("\nBest RF parameters:")
print(rf_search.best_params_)

Training Random Forest...

Best RF parameters:
{'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20}


In [21]:
xgb_param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 9],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb.XGBClassifier(
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    param_distributions=xgb_param_dist,
    n_iter=25,
    cv=3,
    scoring="f1_macro",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("Training XGBoost...")
xgb_search.fit(X_train, y_train)

best_xgb = xgb_search.best_estimator_

print("\nBest XGBoost parameters:")
print(xgb_search.best_params_)

Training XGBoost...

Best XGBoost parameters:
{'subsample': 0.8, 'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.7}


In [22]:
lgb_param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [5, 7, 9, -1],
    "num_leaves": [15, 31, 63, 127],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "min_child_samples": [5, 10, 20]
}

lgb_search = RandomizedSearchCV(
    estimator=lgb.LGBMClassifier(
        verbose=-1,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    param_distributions=lgb_param_dist,
    n_iter=25,
    cv=3,
    scoring="f1_macro",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("Training LightGBM...")
lgb_search.fit(X_train, y_train)

best_lgb = lgb_search.best_estimator_

print("\nBest LightGBM parameters:")
print(lgb_search.best_params_)

Training LightGBM...

Best LightGBM parameters:
{'num_leaves': 31, 'n_estimators': 300, 'min_child_samples': 20, 'max_depth': 7, 'learning_rate': 0.1}


In [24]:
# ============================================================
# SVM BASELINE MODEL
# SVM is used only as a conventional baseline.
# It is NOT included in DA-AMS model selection.
# ============================================================

from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
import time
import joblib
import os

print("Training SVM baseline...")

# ------------------------------------------------------------
# 1. Initialize SVM with fixed, conventional parameters
# ------------------------------------------------------------

svm_baseline = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    class_weight="balanced",
    probability=False,
    cache_size=2048,
    random_state=RANDOM_STATE
)

# ------------------------------------------------------------
# 2. Train SVM
# ------------------------------------------------------------

svm_start = time.perf_counter()

svm_baseline.fit(X_train_scaled, y_train)

svm_train_time = time.perf_counter() - svm_start

print(f"SVM training completed in {svm_train_time:.2f} seconds.")

# ------------------------------------------------------------
# 3. Prediction
# ------------------------------------------------------------

svm_pred_start = time.perf_counter()

svm_pred = svm_baseline.predict(X_test_scaled)

svm_pred_time = time.perf_counter() - svm_pred_start

# ------------------------------------------------------------
# 4. Predictive performance
# ------------------------------------------------------------

svm_accuracy = accuracy_score(y_test, svm_pred)

svm_macro_f1 = f1_score(
    y_test,
    svm_pred,
    average="macro"
)

# decision_function is used because probability=False
svm_decision = svm_baseline.decision_function(X_test_scaled)

svm_roc_auc = roc_auc_score(
    y_test,
    svm_decision
)

# ------------------------------------------------------------
# 5. Per-flow inference latency
# ------------------------------------------------------------

svm_num_flows = len(X_test_scaled)

svm_latency_per_flow = (
    svm_pred_time / svm_num_flows
)

# ------------------------------------------------------------
# 6. Model size
# ------------------------------------------------------------

svm_model_path = "svm_baseline.pkl"

joblib.dump(
    svm_baseline,
    svm_model_path
)

svm_model_size_mb = (
    os.path.getsize(svm_model_path) / (1024 ** 2)
)

# ------------------------------------------------------------
# 7. Display results
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("SVM BASELINE RESULTS")
print("=" * 55)

print(f"Accuracy              : {svm_accuracy:.6f}")
print(f"Macro-F1              : {svm_macro_f1:.6f}")
print(f"ROC-AUC               : {svm_roc_auc:.6f}")
print(f"Training Time (s)     : {svm_train_time:.4f}")
print(f"Total Inference (s)   : {svm_pred_time:.6f}")
print(f"Latency / Flow (s)    : {svm_latency_per_flow:.9f}")
print(f"Model Size (MB)       : {svm_model_size_mb:.6f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        svm_pred,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, svm_pred))

Training SVM baseline...
SVM training completed in 17.38 seconds.

SVM BASELINE RESULTS
Accuracy              : 0.964159
Macro-F1              : 0.960881
ROC-AUC               : 0.979545
Training Time (s)     : 17.3767
Total Inference (s)   : 3.569968
Latency / Flow (s)    : 0.000976736
Model Size (MB)       : 1.351527

Classification Report:
              precision    recall  f1-score   support

           0     0.9871    0.9578    0.9722      2392
           1     0.9243    0.9762    0.9496      1263

    accuracy                         0.9642      3655
   macro avg     0.9557    0.9670    0.9609      3655
weighted avg     0.9654    0.9642    0.9644      3655


Confusion Matrix:
[[2291  101]
 [  30 1233]]


In [31]:
# ============================================================
# DA-AMS CANDIDATE MODEL EVALUATION
# SVM is NOT included because it is a separate baseline.
# ============================================================

candidate_models = {
    "Random Forest": (best_rf, X_val),
    "XGBoost": (best_xgb, X_val),
    "LightGBM": (best_lgb, X_val)
}

validation_results = []

print("=" * 70)
print("DA-AMS CANDIDATE MODEL EVALUATION")
print("=" * 70)

for model_name, (model, X_eval) in candidate_models.items():

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    start_time = time.perf_counter()

    y_pred = model.predict(X_eval)

    end_time = time.perf_counter()

    # Number of flows
    n_flows = len(X_eval)

    # Per-flow inference latency
    latency = (end_time - start_time) / n_flows

    # --------------------------------------------------------
    # Predictive metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_val,
        y_pred
    )

    macro_f1 = f1_score(
        y_val,
        y_pred,
        average="macro"
    )

    # --------------------------------------------------------
    # Correct predictions and errors
    # --------------------------------------------------------

    correct_predictions = int(
        (y_pred == y_val).sum()
    )

    errors = int(
        (y_pred != y_val).sum()
    )

    # --------------------------------------------------------
    # ROC-AUC
    # --------------------------------------------------------

    if hasattr(model, "predict_proba"):

        y_prob = model.predict_proba(X_eval)[:, 1]

        roc_auc = roc_auc_score(
            y_val,
            y_prob
        )

    else:

        y_score = model.decision_function(X_eval)

        roc_auc = roc_auc_score(
            y_val,
            y_score
        )

    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    validation_results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "MacroF1": macro_f1,
        "ROC_AUC": roc_auc,
        "Correct": correct_predictions,
        "Errors": errors,
        "Latency": latency
    })

    # --------------------------------------------------------
    # Display individual results
    # --------------------------------------------------------

    print(f"\n{model_name}")
    print(f"Accuracy          : {accuracy:.6f}")
    print(f"Macro-F1          : {macro_f1:.6f}")
    print(f"ROC-AUC           : {roc_auc:.6f}")
    print(f"Correct           : {correct_predictions}")
    print(f"Errors            : {errors}")
    print(f"Latency/flow      : {latency:.9f} s")


# ============================================================
# Convert results to DataFrame
# ============================================================

validation_df = pd.DataFrame(validation_results)

print("\nDA-AMS candidate evaluation completed.")

DA-AMS CANDIDATE MODEL EVALUATION

Random Forest
Accuracy          : 0.990971
Macro-F1          : 0.990058
ROC-AUC           : 0.999165
Correct           : 3622
Errors            : 33
Latency/flow      : 0.000445552 s

XGBoost
Accuracy          : 0.992339
Macro-F1          : 0.991569
ROC-AUC           : 0.999750
Correct           : 3627
Errors            : 28
Latency/flow      : 0.000078144 s

LightGBM
Accuracy          : 0.991792
Macro-F1          : 0.990960
ROC-AUC           : 0.999740
Correct           : 3625
Errors            : 30
Latency/flow      : 0.000023246 s

DA-AMS candidate evaluation completed.


In [32]:
display_df = validation_df.copy()

display_df["Accuracy"] = (
    display_df["Accuracy"] * 100
)

display_df["MacroF1"] = (
    display_df["MacroF1"] * 100
)

display_df["ROC_AUC"] = (
    display_df["ROC_AUC"] * 100
)

display_df

,Model,Accuracy,MacroF1,ROC_AUC,Correct,Errors,Latency
0,Random Forest,99.097127,99.005759,99.916454,3622,33,0.000446
1,XGBoost,99.233926,99.156863,99.974976,3627,28,0.000078
2,LightGBM,99.179207,99.095979,99.974033,3625,30,0.000023


In [33]:
import joblib
import os

# ============================================================
# SERIALIZED MODEL SIZE
# ============================================================

models_for_size = {
    "Random Forest": best_rf,
    "XGBoost": best_xgb,
    "LightGBM": best_lgb
}

model_sizes = {}

for model_name, model in models_for_size.items():

    filename = model_name.replace(" ", "_").lower() + "_size.pkl"

    joblib.dump(model, filename)

    size_mb = os.path.getsize(filename) / (1024 ** 2)

    model_sizes[model_name] = size_mb

    print(f"{model_name}: {size_mb:.6f} MB")

Random Forest: 5.592919 MB
XGBoost: 0.736051 MB
LightGBM: 0.892506 MB
